# **PYSPARK INTERVIEW QUESTIONS - trinadah**

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

**Q1 While ingesting customer data from an external source, you notice duplicate entries. How would you remove duplicates and retain only the latest entry based on a timestamp column?**

In [0]:
data = [("101", "2023-12-01", 100), ("101", "2023-12-02", 150), 
        ("102", "2023-12-01", 200), ("102", "2023-12-02", 250)]
columns = ["product_id", "date", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
# Casting date col from string to date format
df = df.withColumn('date', col('date').cast(DateType()))

# Drop Duplicates
df = df.orderBy('product_id', 'date', ascending=[1,0]) \
       .dropDuplicates(subset=['product_id'])
df.display()


Note - Spark keeps the first occurrence of each product_id and removes the rest.


**2.**While** preparing a data pipeline, you notice some duplicate rows in a dataset. How would you remove the duplicates without affecting the original order?**

In [0]:
data = [("John", 25), ("Jane", 30), ("John", 25), ("Alice", 22)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# Keep the youngest record for each name
df = df.withColumn(
    "rowflag",
    row_number().over(
        Window.partitionBy("name").orderBy("age")
    )
).filter(col("rowflag") == 1)

df.display()

**3. While processing data from multiple files with inconsistent schemas, you need to merge them into a single DataFrame. How would you handle this inconsistency in PySpark?**

In [0]:
# Read multiple parquet files with different schemas
df = spark.read.format("parquet") \
    .option("mergeSchema", "true") \
    .load("/File/Data/datafiles")

df.show()
#change path it works

**4. You need to process a large dataset stored in PARQUET format and ensure that all columns have the right schema (Almost). How would you do this?**

In [0]:
# Read Parquet data and infer schema
df = spark.read.format("parquet") \
    .option("inferSchema", "true") \
    .load("path")

df.printSchema()
df.display()

**5. While reading data from Parquet, you need to optimize performance by partitioning the data based on a column. How would you implement this?**

In [0]:
# Write Parquet data partitioned by category column
df.write.format("parquet") \
    .mode("append") \
    .partitionBy("category") \
    .save("location")

**4. You are working with a real-time data pipeline, and you notice missing values in your streaming data Column - Category. How would you handle null or missing values in such a scenario?**

**df_stream = spark.readStream.schema("id INT, value STRING").csv("path/to/stream")**

In [0]:
df = df.fillna({'category': 'N/A'})

**5. You need to calculate the total number of actions performed by users in a system. How would you calculate the top 5 most active users based on this information?**

In [0]:
data = [("user1", 5), ("user2", 8), ("user3", 2), ("user4", 10), ("user2", 3)]
columns = ["user_id", "actions"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.functions import sum

# Top 5 users by total actions
df = df.groupBy('user_id') \
       .agg(sum('actions').alias('total_actions')) \
       .orderBy('total_actions', ascending=False) \
       .limit(5)

df.display()

**6. While processing sales transaction data, you need to identify the most recent transaction for each customer. How would you approach this task?**

In [0]:
data = [("cust1", "2023-12-01", 100), ("cust2", "2023-12-02", 150),
        ("cust1", "2023-12-03", 200), ("cust2", "2023-12-04", 250)]
columns = ["customer_id", "transaction_date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, dense_rank
from pyspark.sql.types import DateType

# Cast transaction_date to DateType
df = df.withColumn(
    "transaction_date",
    col("transaction_date").cast(DateType())
)

# Get latest transaction for each customer
df = df.withColumn(
    "flag",
    dense_rank().over(
        Window.partitionBy("customer_id")
              .orderBy(col("transaction_date").desc())
    )
).filter(col("flag") == 1)

df.display()

**7. You need to identify customers who haven’t made any purchases in the last 30 days. How would you filter such customers?**

In [0]:
data = [("cust1", "2025-12-01"), ("cust2", "2024-11-20"), ("cust3", "2024-11-25")]
columns = ["customer_id", "last_purchase_date"]

df = spark.createDataFrame(data, columns)

df.display()

In [0]:
from pyspark.sql.functions import to_date, datediff, current_date, col

# Convert last_purchase_date to date format
df = df.withColumn(
    "last_purchase_date",
    to_date(col("last_purchase_date"))
)

# Calculate gap in days and filter customers inactive for more than 30 days
df = df.withColumn(
    "gap",
    datediff(current_date(), col("last_purchase_date"))
).filter(col("gap") > 30)

df.display()

**8. While analyzing customer reviews, you need to identify the most frequently used words in the feedback. How would you implement this?**

In [0]:
data = [("customer1", "The product is great"), ("customer2", "Great product, fast delivery"), ("customer3", "Not bad, could be better")]
columns = ["customer_id", "feedback"]

df = spark.createDataFrame(data, columns)

df.display()

In [0]:
from pyspark.sql.functions import explode, split

# Split feedback into words
df = df.withColumn(
    "feedback",
    explode(split("feedback", " "))
)

df.display()

In [0]:
from pyspark.sql.functions import explode, split, count

df = df.withColumn(
    "feedback",
    explode(split("feedback", " "))
)

df_grp = df.groupBy("feedback") \
           .agg(count("feedback").alias("wordcount"))

df_grp.display()

In [0]:
from pyspark.sql.functions import explode, split, count, lower

df = df.withColumn("feedback", lower("feedback")) \
       .withColumn("feedback", explode(split("feedback", " ")))

df_grp = df.groupBy("feedback") \
           .agg(count("feedback").alias("wordcount"))

df_grp.display()

**9. You need to calculate the cumulative sum of sales over time for each product. How would you approach this?**

In [0]:
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-02", 200),
        ("product1", "2023-12-03", 150), ("product2", "2023-12-04", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, to_date

# Convert date column to DateType
df = df.withColumn("date", to_date(col("date")))

# Calculate cumulative sales for each product
windowSpec = Window.partitionBy("product_id").orderBy("date")

df = df.withColumn(
    "CumSum",
    sum("sales").over(windowSpec)
)

df.display()

**10. You are working with user activity data and need to calculate the average session duration per user. How would you implement this?**

In [0]:
data = [("user1", "2023-12-01", 50), ("user1", "2023-12-02", 60), 
        ("user2", "2023-12-01", 45), ("user2", "2023-12-03", 75)]
columns = ["user_id", "session_date", "duration"]
df = spark.createDataFrame(data, columns)

df.display()

In [0]:
from pyspark.sql.functions import avg

# Calculate average duration for each user
df = df.groupBy("user_id") \
       .agg(avg("duration").alias("avg_duration"))

df.display()

**12. While analyzing sales data, you need to find the product with the highest sales for each month. How would you accomplish this?**

In [0]:
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-01", 150), 
        ("product1", "2023-12-02", 200), ("product2", "2023-12-02", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.functions import to_date, month, sum

# Convert date column to DateType
df = df.withColumn("date", to_date("date"))

# Calculate monthly sales by product
df = df.withColumn("date", month("date")) \
       .groupBy("date", "product_id") \
       .agg(sum("sales").alias("total_sales"))

df.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import to_date, month, sum, dense_rank, col

# Convert date column to DateType
df = df.withColumn("date", to_date("date"))

# Calculate total sales by month and product
df = df.withColumn("date", month("date")) \
       .groupBy("date", "product_id") \
       .agg(sum("sales").alias("sales"))

# Get top-selling product for each month
df = df.withColumn(
    "ranking",
    dense_rank().over(
        Window.partitionBy("date")
              .orderBy(col("sales").desc())
    )
).filter(col("ranking") == 1)

df.display()

**13. You are working with a large Delta table that is frequently updated by multiple users. The data is stored in partitions, and sometimes updates can cause inconsistent reads due to concurrent transactions. How would you ensure ACID compliance and avoid data corruption in PySpark?**

In [0]:
from delta.tables import DeltaTable

# Read new data
df = spark.read.format("parquet") \
    .load("path")

# Load target Delta table
delta_tbl = DeltaTable.forPath(spark, "path")

# Perform UPSERT (Merge)
delta_tbl.alias("trg") \
    .merge(
        df.alias("src"),
        "src.id = trg.id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

note----When people inserting into data on same table …it will creats duplictaes ,so use scd

**15. You are reading a CSV file and need to handle corrupt records gracefully by skipping them. How would you configure this in PySpark?**

In [0]:
# Read CSV file and skip corrupt/malformed records
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("mode", "DROPMALFORMED") \
    .load("staging_location")

df.display()
#Other Modes
# Fail if malformed records exist
.option("mode", "FAILFAST")
# Keep malformed records as null (default)
.option("mode", "PERMISSIVE")

**22. You have a dataset containing the names of employees and their departments. You need to find the department with the most employees.**

In [0]:
data = [("Alice", "HR"), ("Bob", "Finance"), ("Charlie", "HR"), ("David", "Engineering"), ("Eve", "Finance")]
columns = ["employee_name", "department"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.functions import count

# Count employees in each department
df = df.groupBy("department") \
       .agg(count("employee_name").alias("total_employees")) \
       .sort("total_employees", ascending=False)

df.display()

**23. While processing sales data, you need to classify each transaction as either 'High' or 'Low' based on its amount. How would you achieve this using a when condition**

In [0]:
data = [("product1", 100), ("product2", 300), ("product3", 50)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.functions import when, col

# Categorize products based on sales
df = df.withColumn(
    "price_cat",
    when(col("sales") > 50, "High")
    .otherwise("Low")
)

df.display()

**24. While analyzing a large dataset, you need to create a new column that holds a timestamp of when the record was processed. How would you implement this and what can be the best USE CASE?**

In [0]:
data = [("product1", 100), ("product2", 200), ("product3", 300)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.functions import current_timestamp

# Add current processing timestamp
df = df.withColumn(
    "processed_time",
    current_timestamp()
)

df.display()

**25. You need to register this PySpark DataFrame as a temporary SQL object and run a query on it. How would you achieve this?**

In [0]:
data = [("product1", 100), ("product2", 200), ("product3", 300)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
# Create a temporary SQL view from DataFrame
df.createOrReplaceTempView("tempsqldf")

In [0]:
# Select all records
spark.sql("""
SELECT *
FROM tempsqldf
""").display()

In [0]:
spark.sql("""
SELECT *
FROM tempsqldf
WHERE product_id = 'product1'
""").display()

**26. You need to register this PySpark DataFrame as a temporary SQL object and run a query on it (FROM DIFFERENT NOTEBOOKS AS WELL)?**

In [0]:
# Create a Global Temporary View
df.createOrReplaceGlobalTempView("globalview")

In [0]:
%sql
SELECT *
FROM global_temp.globalview;

In [0]:
spark.sql("""
SELECT *
FROM global_temp.globalview
""").display()

**27. You need to query data from a PySpark DataFrame using SQL, but the data includes a nested structure. How would you flatten the data for easier querying?**

In [0]:
data = [("product1", {"price": 100, "quantity": 2}), 
        ("product2", {"price": 200, "quantity": 3})]
columns = ["product_id", "product_info"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
# Select nested fields and create a temporary view
df.select(
    "product_id",
    "product_info.price",
    "product_info.quantity"
).createOrReplaceTempView("flatview")

In [0]:
%sql
SELECT *
FROM flatview;


**28. You are ingesting data from an external API in JSON format where the schema is inconsistent. How would you handle this situation to ensure a robust pipeline?**

In [0]:
from pyspark.sql.functions import col

# Read JSON data
df = spark.read.option("multiLine", "true") \
    .option("mergeSchema", "true") \
    .json("path")

# Select required columns and handle missing fields
df = df.select(
    col("id"),
    col("name"),
    col("amount").cast("double")
)

df.display()

**30. You are working with a large dataset in Parquet format and need to ensure that the data is written in an optimized manner with proper compression. How would you accomplish this?**

In [0]:
# Write Parquet data with Snappy compression
df.write.format("parquet") \
    .option("compression", "snappy") \
    .mode("overwrite") \
    .save("location")

**31. Your company uses a large-scale data pipeline that reads from Delta tables and processes data using complex aggregations. However, performance is becoming an issue due to the growing dataset size. How would you optimize the performance of the pipeline?**

In [0]:
%sql
OPTIMIZE tabledelta
ZORDER BY (order_date)

**43. You are processing sales data. Group by product categories and create a list of all product names in each category.**

In [0]:
data = [("Electronics", "Laptop"), ("Electronics", "Smartphone"), ("Furniture", "Chair"), ("Furniture", "Table")]
columns = ["category", "product"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.functions import collect_list

# Collect products into an array for each category
df = df.groupBy("category") \
       .agg(collect_list("product").alias("products"))

df.display()

**44. You are analyzing orders. Group by customer IDs and list all unique product IDs each customer purchased.**

In [0]:
data = [(101, "P001"), (101, "P002"), (102, "P001"), (101, "P001")]
columns = ["customer_id", "product_id"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.functions import collect_set

# Collect unique products for each customer
df = df.groupBy("customer_id") \
       .agg(collect_set("product_id").alias("unique_products"))

df.display()


**45. For customer records, combine first and last names only if the email address exists.**

In [0]:
data = [("John", "Doe", "john.doe@example.com"), ("Jane", "Smith", None)]
columns = ["first_name", "last_name", "email"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.functions import when, col, concat_ws

# Create fullname only when email is not null
df = df.withColumn(
    "fullname",
    when(
        col("email").isNotNull(),
        concat_ws("-", col("first_name"), col("last_name"))
    ).otherwise(None)
)

df.display()

**46. You have a DataFrame containing customer IDs and a list of their purchased product IDs. Calculate the number of products each customer has purchased.**

In [0]:
data = [
    (1, ["prod1", "prod2", "prod3"]),
    (2, ["prod4"]),
    (3, ["prod5", "prod6"]),
]
myschema = "customer_id INT ,product_ids array<STRING>"

df = spark.createDataFrame(data, myschema)
df.display()

In [0]:
from pyspark.sql.functions import size, col

# Count number of products in the array column
df = df.withColumn(
    "number_of_products",
    size(col("product_ids"))
)

df.display()

**47. You have employee IDs of varying lengths. Ensure all IDs are 6 characters long by padding with leading zeroes.**

In [0]:
data = [
    ("1",),
    ("123",),
    ("4567",),
]
schema = ["employee_id"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
from pyspark.sql.functions import lpad, col

# Pad employee_id with leading zeros to make length = 6
df = df.withColumn(
    "employee_id",
    lpad(col("employee_id"), 6, "0")
)

df.display()

**48. You need to validate phone numbers by checking if they start with "91"**

In [0]:
data = [
    ("911234567890",),
    ("811234567890",),
    ("912345678901",),
]
schema = ["phone_number"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
from pyspark.sql.functions import substring, col

# Filter phone numbers starting with 91
df.filter(
    substring(col("phone_number"), 1, 2) == "91"
).display()

**49. You have a dataset with courses taken by students. Calculate the average number of courses per student.**

In [0]:
data = [
    (1, ["Math", "Science"]),
    (2, ["History"]),
    (3, ["Art", "PE", "Biology"]),
]
schema = ["student_id", "courses"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
from pyspark.sql.functions import size, avg

# Calculate average number of courses per student
df = df.withColumn(
    "course_size",
    size("courses")
).groupBy() \
 .agg(avg("course_size").alias("avg_course_size"))

df.display()

**50. You have a dataset with primary and secondary contact numbers. Use the primary number if available; otherwise, use the secondary number.**

In [0]:
data = [
    (None, "1234567890"),
    ("9876543210", None),
    ("7894561230", "4567891230"),
]
schema = ["primary_contact", "secondary_contact"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
from pyspark.sql.functions import coalesce, col

# Get contact number from primary_contact or secondary_contact
df = df.withColumn(
    "contact",
    coalesce(
        col("primary_contact"),
        col("secondary_contact")
    )
)

df.display()

**51. You are categorizing product codes based on their lengths. If the length is 5, label it as "Standard"; otherwise, label it as "Custom".**

In [0]:
data = [
    ("prod1",),
    ("prd234",),
    ("pr9876",),
]
schema = ["product_code"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
from pyspark.sql.functions import when, length, col

# Categorize product codes based on length
df = df.withColumn(
    "Code_Flag",
    when(length(col("product_code")) == 5, "Standard")
    .otherwise("Custom")
)

df.display()